In [70]:
import pandas as pd
import re
import csv
import re
import sys
from collections import Counter, defaultdict

df_extract = pd.read_csv(
    "../data/interim/extract_15_16_concat.csv",
    low_memory=False,
)
df_extract.shape

(1127838, 29)

In [84]:
df_ND1516 = pd.read_csv(
    "../data/raw/test_regards_citoyens/ND15+16_interventions_hemicycle_rich.tsv",
    sep="\t",
    engine="python",
    on_bad_lines="warn",
)

df_ND1516.shape

(1391207, 16)

In [85]:
df_ND1516.columns

Index(['id', 'seance_id', 'date', 'moment', 'type', 'section', 'sous_section',
       'timestamp', 'intervention', 'nb_mots', 'personnalite', 'parlementaire',
       'parlementaire_sexe', 'parlementaire_groupe', 'fonction', 'source'],
      dtype='object')

## version simple

In [88]:
# === Comparaison simple par id_syceron ===

# extraite equivalent id_syceron vs url Pnum
P_NUMBER_RE = re.compile(r"#P(\d+)")


def extract_pnum(url):
    if not url:
        return None
    m = P_NUMBER_RE.search(url)
    return m.group(1) if m else None


# Colonnes normalisées
df_extract["id_syceron"] = df_extract["id_syceron"].dropna().astype(float).astype(int)
df_ND1516["pnum"] = df_ND1516["source"].apply(extract_pnum).dropna().astype(int)

# Comparaison sur les colonnes normalisées
ids_extract = set(df_extract["id_syceron"].dropna().astype(int))
ids_ND = set(df_ND1516["pnum"].dropna().astype(int))

common = ids_extract & ids_ND
only_extract = ids_extract - ids_ND
only_ND = ids_ND - ids_extract


print("=== RÉSUMÉ GLOBAL (par id_syceron uniquement) ===")
print(f"IDs communs              : {len(common):>10,}")
print(f"Uniquement dans extract  : {len(only_extract):>10,}")
print(f"Uniquement dans ND15-16  : {len(only_ND):>10,}")

print(f"\nTotal IDs extract : {len(ids_extract):>10,}")
print(f"Total IDs ND      : {len(ids_ND):>10,}")

# Inspection des divergences — isin() sur la colonne déjà normalisée
print("=== Dans extract mais PAS dans ND ===")
display(df_extract[df_extract["id_syceron"].isin(only_extract)].head(5))

print("=== Dans ND mais PAS dans extract ===")
display(df_ND1516[df_ND1516["pnum"].isin(only_ND)].head(5))

=== RÉSUMÉ GLOBAL (par id_syceron uniquement) ===
IDs communs              :    603,663
Uniquement dans extract  :    523,808
Uniquement dans ND15-16  :     25,796

Total IDs extract :  1,127,471
Total IDs ND      :    629,459
=== Dans extract mais PAS dans ND ===


,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,code_grammaire,code_style,code_parole,id_syceron,roledebat,nom_orateur,qualite_orateur,id_orateur,stime,texte
1,CRSANR5L15S2017E1N001,NaN,NaN,20170704,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,OUV_SEAN_2_2,Info Italiques,NaN,981339,NaN,NaN,NaN,NaN,NaN,(La séance est ouverte à quinze heures.)
255,CRSANR5L15S2017E1N001,NaN,NaN,20170704,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,SUSP_SEANCE_2_2,Info Italiques,NaN,982051,NaN,NaN,NaN,NaN,NaN,"(La séance, suspendue à dix-huit heures quinze..."
259,CRSANR5L15S2017E1N001,NaN,NaN,20170704,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,FIN_SEAN_2_4,Info Italiques,NaN,982060,NaN,NaN,NaN,NaN,NaN,(La séance est levée à dix-huit heures cinquan...
260,CRSANR5L15S2017E1N001,NaN,NaN,20170704,mardi 04 juillet 2017,Unique,1,AN,15,Première session extraordinaire 2017,...,FIN_SEAN_2_4,Info Italiques,NaN,982060,NaN,NaN,NaN,NaN,NaN,La Directrice du service du compte rendu de la...
262,CRSANR5L15S2017E1N002,NaN,NaN,20170705,mercredi 05 juillet 2017,Unique,2,AN,15,Première session extraordinaire 2017,...,OUV_SEAN_2_2,Info Italiques,NaN,982176,NaN,NaN,NaN,NaN,NaN,(La séance est ouverte à quinze heures.)


=== Dans ND mais PAS dans extract ===


,id,seance_id,date,moment,type,section,sous_section,timestamp,intervention,nb_mots,personnalite,parlementaire,parlementaire_sexe,parlementaire_groupe,fonction,source,id_syceron,pnum
2,3,1,2017-06-27,15:00,loi,ouverture de la xve législature,ouverture de la xve législature,60,<p>Ouverture de la XVe législature</p>,8,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980119,980119.0
4,5,1,2017-06-27,15:00,loi,constitution du bureau d'âge,constitution du bureau d'âge,80,<p>constitution du bureau d'âge</p>,7,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980122,980122.0
6,7,1,2017-06-27,15:00,loi,communication de la liste des députés,communication de la liste des députés,100,<p>communication de la liste des députés</p>,10,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980125,980125.0
8,9,1,2017-06-27,15:00,loi,députés nommés membres du gouvernement,députés nommés membres du gouvernement,120,<p>députés nommés membres du gouvernement</p>,10,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980128,980128.0
10,11,1,2017-06-27,15:00,loi,décès de deux députés de la xive législature,décès de deux députés de la xive législature,140,<p>décès de deux députés de la xive législatur...,15,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...,980131,980131.0


In [100]:
# Vérification sur un échantillon aléatoire des IDs communs
sample_common = pd.Series(list(common)).sample(10, random_state=42)

df_sample_extract = df_extract[df_extract["id_syceron"].isin(sample_common)][
    ["id_syceron", "texte"]
].rename(columns={"id_syceron": "id"})  # ← adapter si la colonne texte a un autre nom

df_sample_ND = df_ND1516[df_ND1516["pnum"].isin(sample_common)][
    ["pnum", "intervention"]
].rename(columns={"pnum": "id"})  # ← adapter si la colonne intervention a un autre nom

# Fusion sur l'id commun
df_check = df_sample_extract.merge(df_sample_ND, on="id")

display(df_check)

print(
    "LÉO, NORMAL QUE TU AIES DES 'DOUBLONS'",
    "\nTON FICHIER MARCHE C'EST À CAUSE  DU CHANGEMENT TEXTE DANS ND)",
)

,id,texte,intervention
0,986459,Je remercie mon collègue André Chassaigne d’av...,<p>Je remercie mon collègue André Chassaigne d...
1,1107128,J’ai un peu de recul sur cette question. J’ai ...,<p>J'ai un peu de recul sur cette question. J'...
2,1145704,"Cet article, adopté par le Sénat, était relati...","<p>Cet article, adopté par le Sénat, était rel..."
3,1145704,"Cet article, adopté par le Sénat, était relati...","<p>L'amendement no 402, accepté par le Gouvern..."
4,1351691,"La parole est à M. Bastien Lachaud, pour le gr...","<p>La parole est à M. Bastien Lachaud, pour le..."
5,1564711,Je profite de cette intervention sur l’article...,<p>Je profite de cette intervention sur l'arti...
6,1564711,Je profite de cette intervention sur l’article...,<p>Mme Huguette Bello et Mme Maina Sage applau...
7,1689014,Cela fait deux ans que vous siégez sur ces ban...,<p>Cela fait deux ans que vous siégez sur ces ...
8,2031391,La suite de la discussion est renvoyée à la pr...,<p>La suite de la discussion est renvoyée à la...
9,2031391,La suite de la discussion est renvoyée à la pr...,<p>Exclamations et applaudissements. </p>


LÉO, NORMAL QUE TU AIES DES 'DOUBLONS' 
TON FICHIER MARCHE C'EST À CAUSE  DU CHANGEMENT TEXTE DANS ND)


Cet amendement n’a pas été étudié par la commission des finances ; à titre personnel, j’y suis favorable.

vs
1 = <p>Cet amendement n'a pas été étudié par la commission des finances ; à titre personnel, j'y suis favorable.</p>
2 = <p>L'amendement no 2443 est adopté. </p>
3 = <p>Les crédits du compte de concours financiers « Prêts et avances à des particuliers ou à des organismes privés », modifiés, sont adoptés. </p>

OU :
Je profite de cette intervention sur l’article 1er pour exprimer le soutien fraternel de la Corse à la proposition de loi, aboutissement, dans un cadre de décision démocratique, d’un processus que M. Letchimy a fort bien retracé.Il va de soi que nous voterons ce texte, qui est l’émanation d’une ascendance culturelle, géographique, territoriale, et qui prend en compte une différence.En Corse, il y a autant de biens en indivision qu’en Martinique, et ceux-ci sont possédés par plusieurs générations. Dans le système territorial insulaire – mais pas uniquement dans celui-ci –, la rareté foncière, soumise à l’indivision et à la spéculation immobilière, crée des ruptures sociales, économiques et culturelles, parce que le lien à la terre s’est construit, au fil des générations, sur les notions de survivance et d’autarcie.Cette situation ne peut évidemment être traitée que par la différenciation des solutions. C’est pourquoi je salue, au-delà de l’article 1er, la construction effectuée dans ce texte. Peut-être entamons-nous aujourd’hui une longue marche qui nous mènera vers une relation différente à la République.C’est du moins ce que nous souhaitons, et c’est pour cette raison, monsieur le rapporteur, madame la ministre, que je tenais à exprimer ma gratitude. (Mme Huguette Bello et Mme Maina Sage applaudissent.)

vs
1 = <p>Je profite de cette intervention sur l'article 1er pour exprimer le soutien fraternel de la Corse à la proposition de loi, aboutissement, dans un cadre de décision démocratique, d'un processus que M. Letchimy a fort bien retracé.</p><p>Il va de soi que nous voterons ce texte, qui est l'émanation d'une ascendance culturelle, géographique, territoriale, et qui prend en compte une différence.</p><p>En Corse, il y a autant de biens en indivision qu'en Martinique, et ceux-ci sont possédés par plusieurs générations. Dans le système territorial insulaire – mais pas uniquement dans celui-ci – , la rareté foncière, soumise à l'indivision et à la spéculation immobilière, crée des ruptures sociales, économiques et culturelles, parce que le lien à la terre s'est construit, au fil des générations, sur les notions de survivance et d'autarcie.</p><p>Cette situation ne peut évidemment être traitée que par la différenciation des solutions. C'est pourquoi je salue, au-delà de l'article 1er, la construction effectuée dans ce texte. Peut-être entamons-nous aujourd'hui une longue marche qui nous mènera vers une relation différente à la République.</p><p>C'est du moins ce que nous souhaitons, et c'est pour cette raison, monsieur le rapporteur, madame la ministre, que je tenais à exprimer ma gratitude.</p>
2 = <p>Mme Huguette Bello et Mme Maina Sage applaudissent. </p>

In [102]:
df_check

,id,texte,intervention
0,986459,Je remercie mon collègue André Chassaigne d’av...,<p>Je remercie mon collègue André Chassaigne d...
1,1107128,J’ai un peu de recul sur cette question. J’ai ...,<p>J'ai un peu de recul sur cette question. J'...
2,1145704,"Cet article, adopté par le Sénat, était relati...","<p>Cet article, adopté par le Sénat, était rel..."
3,1145704,"Cet article, adopté par le Sénat, était relati...","<p>L'amendement no 402, accepté par le Gouvern..."
4,1351691,"La parole est à M. Bastien Lachaud, pour le gr...","<p>La parole est à M. Bastien Lachaud, pour le..."
5,1564711,Je profite de cette intervention sur l’article...,<p>Je profite de cette intervention sur l'arti...
6,1564711,Je profite de cette intervention sur l’article...,<p>Mme Huguette Bello et Mme Maina Sage applau...
7,1689014,Cela fait deux ans que vous siégez sur ces ban...,<p>Cela fait deux ans que vous siégez sur ces ...
8,2031391,La suite de la discussion est renvoyée à la pr...,<p>La suite de la discussion est renvoyée à la...
9,2031391,La suite de la discussion est renvoyée à la pr...,<p>Exclamations et applaudissements. </p>


In [ ]:
# en fait non
list_pb = [
    1145704,
    1145704,
    1564711,
    1564711,
    2031391,
    2031391,
    2290890,
    2290890,
    2290890,
]

pb = df_extract[df_extract["id_syceron"].isin(list_pb)]

In [97]:
pb

,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,code_grammaire,code_style,code_parole,id_syceron,roledebat,nom_orateur,qualite_orateur,id_orateur,stime,texte
115906,CRSANR5L15S2018O1N101,NaN,NaN,20171215,vendredi 15 décembre 2017,3,101,AN,15,Session ordinaire 2017-2018,...,DISC_ARTICLES_3_4_1,NORMAL,NaN,1145704,NaN,M. Joël Giraud,rapporteur général,267336.0,NaN,"Cet article, adopté par le Sénat, était relati..."
290355,CRSANR5L15S2019O1N104,NaN,NaN,20181212,mercredi 12 décembre 2018,1,104,AN,15,Session ordinaire 2018-2019,...,DISC_ARTICLES_1_20,NORMAL,PAROLE_1_2,1564711,NaN,M. Jean-Félix Acquaviva,NaN,719146.0,NaN,Je profite de cette intervention sur l’article...
497518,CRSANR5L15S2020O1N157,NaN,NaN,20200222,samedi 22 février 2020,2,157,AN,15,session ordinaire 2019-2020,...,FIN_SEAN_1_0,NORMAL,NaN,2031391,president,M. le président,NaN,720746.0,NaN,La suite de la discussion est renvoyée à la pr...
591082,CRSANR5L15S2021O1N066,NaN,NaN,20201107,samedi 07 novembre 2020,1,66,AN,15,session ordinaire 2020-2021,...,PAROLE_GENERIQUE,NORMAL,AVIS_COM_1_20,2290890,NaN,M. Xavier Roseren,rapporteur spécial,721458.0,NaN,Cet amendement n’a pas été étudié par la commi...


In [ ]:
## Version avec commbinaison ID et date

In [78]:
# FILE1_DATE_COL  = "dateSeance"   # format 20170704150000000
# FILE1_KEY_COL   = "id_syceron"
# FILE2_DATE_COL  = "date"         # format 2017-07-04
# FILE2_SOURCE_COL = "source"      # contient ...#P980116

# P_NUMBER_RE = re.compile(r"#P(\d+)")

# # extraite equivalent id_syceron vs url Pnum
# def extract_pnum(url):
#     if not url:
#         return None
#     m = P_NUMBER_RE.search(url)
#     return m.group(1) if m else None

# # === Normalisation des dates ===

# def normaliser_date(raw):
#     """Ramène tout format de date à YYYYMMDD (str 8 chars)."""
#     if pd.isna(raw):
#         return "UNKNOWN"
#     raw = str(raw).strip()
#     # format 20170704150000000 ou 20170704
#     raw = raw.replace("-", "")  # format 2017-07-04 → 20170704
#     return raw[:8]

# df_extract["dateSeance"] = df_extract["dateSeance"].apply(normaliser_date)
# df_ND1516["date"]  = df_ND1516["date"].apply(normaliser_date)


# # === Construction des index de clés ===

# def build_keys(df, date_col, key_extractor):
#     """
#     Construit les index de clés à partir d'un DataFrame déjà chargé.
#     Retourne :
#       - rows_per_date  : Counter(date -> nb lignes)
#       - keys_per_date  : dict(date -> set(cle))
#       - all_keys       : set((date, cle))
#     """
#     rows_per_date = Counter()
#     keys_per_date = defaultdict(set)
#     all_keys = set()

#     for _, row in df.iterrows():
#         raw_date = str(row.get(date_col, "") or "")
#         date = raw_date.replace("-", "")[:8] if raw_date else "UNKNOWN"
#         rows_per_date[date] += 1
#         key = key_extractor(row)
#         if key:
#             keys_per_date[date].add(key)
#             all_keys.add((date, key))

#     return rows_per_date, keys_per_date, all_keys

# print("Construction des index...")
# rows1, keys1, all_keys1 = build_keys(
#     df_extract,
#     FILE1_DATE_COL,
#     lambda r: r.get(FILE1_KEY_COL) or None,
# )

# rows2, keys2, all_keys2 = build_keys(
#     df_ND1516,
#     FILE2_DATE_COL,
#     lambda r: extract_pnum(r.get(FILE2_SOURCE_COL)),
# )

# print(f"df_extract : {len(df_extract):,} lignes | {len(all_keys1):,} unités (date+id) uniques")
# print(f"df_ND1516  : {len(df_ND1516):,} lignes | {len(all_keys2):,} unités (date+id) uniques")



Construction des index...
df_extract : 1,127,838 lignes | 1,127,471 unités (date+id) uniques
df_ND1516  : 1,391,207 lignes | 629,459 unités (date+id) uniques


In [72]:
# # === Comparaison globale ===

# common = all_keys1 & all_keys2
# only1  = all_keys1 - all_keys2
# only2  = all_keys2 - all_keys1

# print("=== RÉSUMÉ GLOBAL ===")
# print(f"Unités communes                : {len(common):>10,}")
# print(f"Uniquement dans extract        : {len(only1):>10,}")
# print(f"Uniquement dans ND15-16        : {len(only2):>10,}")

=== RÉSUMÉ GLOBAL ===
Unités communes                :          0
Uniquement dans extract        :  1,127,471
Uniquement dans ND15-16        :    629,459


In [73]:
# # === Détail par jour ===

# all_dates = sorted(set(rows1) | set(rows2))
# print(f"{'date':10} {'f1_lignes':>10} {'f1_unites':>10} {'f2_lignes':>10} {'f2_unites':>10} {'ecart':>8}")
# for d in all_dates:
#     r1 = rows1.get(d, 0)
#     u1 = len(keys1.get(d, set()))
#     r2 = rows2.get(d, 0)
#     u2 = len(keys2.get(d, set()))
#     ecart = u2 - u1
#     flag = "  ← absent f1" if r1 == 0 else ("  ← absent f2" if r2 == 0 else "")
#     print(f"{d:10} {r1:>10,} {u1:>10,} {r2:>10,} {u2:>10,} {ecart:>8}{flag}")

date        f1_lignes  f1_unites  f2_lignes  f2_unites    ecart
20170627           26         25         51         26        1
20170628          123        122        189        116       -6
20170703            0          0         95         44       44  ← absent f1
20170704          261        260        592        261        1
20170705          153        152        249        168       16
20170706          500        498        660        487      -11
20170710          759        757      1,060        717      -40
20170711        1,084      1,082      1,353        994      -88
20170712        1,176      1,174      1,531      1,135      -39
20170713        1,282      1,280      1,635      1,217      -63
20170718          853        851      1,065        818      -33
20170719          569        568        800        579       11
20170720          419        418        560        420        2
20170724        1,112      1,110      1,491      1,112        2
20170725        1,320      

In [74]:
# === Inspection et export des divergences ===

df_only1 = df_extract[
    df_extract.apply(
        lambda r: (
            str(r.get(FILE1_DATE_COL, "") or "").replace("-", "")[:8],
            str(r.get(FILE1_KEY_COL, "") or ""),
        )
        in only1,
        axis=1,
    )
]

df_only2 = df_ND1516[
    df_ND1516.apply(
        lambda r: (
            str(r.get(FILE2_DATE_COL, "") or "").replace("-", "")[:8],
            extract_pnum(r.get(FILE2_SOURCE_COL)) or "",
        )
        in only2,
        axis=1,
    )
]

print(f"Lignes uniquement dans extract  : {len(df_only1):,}")
display(df_only1.head(5))

print(f"\nLignes uniquement dans ND15-16  : {len(df_only2):,}")
display(df_only2.head(5))

# Export CSV
df_only1.to_csv("../data/temp/unites_seulement_extract.csv", index=False)
df_only2.to_csv("../data/temp/unites_seulement_ND1516.csv", index=False)
print(
    "\nExports écrits dans unites_seulement_extract.csv et unites_seulement_ND1516.csv"
)

Lignes uniquement dans extract  : 0


,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,code_grammaire,code_style,code_parole,id_syceron,roledebat,nom_orateur,qualite_orateur,id_orateur,stime,texte



Lignes uniquement dans ND15-16  : 796,863


,id,seance_id,date,moment,type,section,sous_section,timestamp,intervention,nb_mots,personnalite,parlementaire,parlementaire_sexe,parlementaire_groupe,fonction,source
0,1,1,2017-06-27,15:00,loi,NaN,NaN,20,<p>La séance est ouverte.</p>,7,NaN,Bernard Brochand,H,NI,"président, doyen d'âge",http://www.assemblee-nationale.fr/15/cri/2016-...
1,2,1,2017-06-27,15:00,loi,NaN,NaN,30,<p>La séance est ouverte à quinze heures.</p>,9,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...
2,3,1,2017-06-27,15:00,loi,ouverture de la xve législature,ouverture de la xve législature,60,<p>Ouverture de la XVe législature</p>,8,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...
3,4,1,2017-06-27,15:00,loi,ouverture de la xve législature,ouverture de la xve législature,70,<p>Je déclare ouverte la XVe législature de l'...,14,NaN,Bernard Brochand,H,NI,"président, doyen d'âge",http://www.assemblee-nationale.fr/15/cri/2016-...
4,5,1,2017-06-27,15:00,loi,constitution du bureau d'âge,constitution du bureau d'âge,80,<p>constitution du bureau d'âge</p>,7,NaN,NaN,NaN,NaN,NaN,http://www.assemblee-nationale.fr/15/cri/2016-...



Exports écrits dans unites_seulement_extract.csv et unites_seulement_ND1516.csv


In [80]:
# Diagnostic rapide
print("Type id_syceron extract :", df_extract["id_syceron"].dtype)
print("Exemple extract :", df_extract["id_syceron"].dropna().iloc[0])
print("Exemple ND pnum :", extract_pnum(df_ND1516["source"].dropna().iloc[0]))

# Vérification : les clés (date, id) sont-elles comparables ?
sample_key1 = list(all_keys1)[:3]
sample_key2 = list(all_keys2)[:3]
print("\nExemples all_keys1 :", sample_key1)
print("Exemples all_keys2 :", sample_key2)

Type id_syceron extract : int64
Exemple extract : 981338
Exemple ND pnum : 980116

Exemples all_keys1 : [('20191007', 1850899), ('20211028', 2654102), ('20191126', 1941389)]
Exemples all_keys2 : [('20201102', '2275179'), ('20171114', '1094541'), ('20171004', '1032804')]


In [81]:
df_extract["id_syceron"].dtype

dtype('int64')

In [83]:
df_ND1516["id_syceron"].dtype

dtype('O')